### Convert data production coffee to CSV

In [ ]:
import pandas as pd
import os
import glob

def extract_year_data(filepath):
    year = int(os.path.basename(filepath).replace('.xlsx', ''))
    df = pd.read_excel(filepath, header=None)

    # Filter only actual village data rows (column 4 == 'Kopi Arabika')
    data = df[df[4] == 'Kopi Arabika'].copy()

    result = pd.DataFrame({
        'year':        year,
        'village':     data[3].str.strip(),
        'tm_ha':       data[6],
        'produksi_kg': data[10]
    })

    return result

# Process all xlsx files in the same folder as this script
files = sorted(glob.glob('../data/*.xlsx'))
print(f"Found files: {files}")

all_data = pd.concat([extract_year_data(f) for f in files], ignore_index=True)

all_data.to_csv('../data/coffeeProduction_benerMeriah.csv', index=False)

print(f"Total records : {len(all_data)}")
print(f"Years         : {sorted(all_data['year'].unique())}")
print(f"Villages      : {all_data['village'].nunique()} unique")
print(all_data.head(10))

Found files: ['../data/2021.xlsx', '../data/2022.xlsx', '../data/2023.xlsx', '../data/2024.xlsx', '../data/2025.xlsx']
Total records : 1100
Years         : [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Villages      : 219 unique
   year           village   tm_ha produksi_kg
0  2021            Kenine  103.02    67601.87
1  2021     Fajar Harapan   21.36    14732.88
2  2021      Damaran Baru  174.01    133971.1
3  2021  Pantan Pediangan  285.18   191159.97
4  2021   Bandar Lampahan   43.13     28048.9
5  2021           Rembune   64.88     48185.4
6  2021       Mude Benara   137.3     94597.1
7  2021          Bumi Ayu   79.07    59300.45
8  2021       Karang Jadi   85.71    64958.71
9  2021   Kampung Baru 76  360.64   234782.69


### Create manual calculation with excel
The formula have some issues due to difference in raw data and excel format

In [1]:
"""
Build: coffee_yield_pipeline.xlsx
9 sheets — fully formula-driven except raw inputs
"""
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()

# ── palette ─────────────────────────────────────────────────────────────────
def fill(hex6): return PatternFill("solid", start_color=hex6)

F_NAVY   = fill("1F4E79")   # dark blue  — section headers
F_TEAL   = fill("0E6655")   # dark green — derived/auto
F_ORANGE = fill("784212")   # brown-orange — interactions
F_PURPLE = fill("4A235A")   # purple — target / model output
F_DKGREY = fill("424949")   # dark grey — structural identifiers

F_INP    = fill("D6EAF8")   # light blue  — user input cells
F_FORM   = fill("D5F5E3")   # light green — formula cells
F_QFORM  = fill("FEF9E7")   # light yellow — quarterly formula cells
F_IFORM  = fill("FDEBD0")   # light orange — interaction formula cells
F_PFORM  = fill("EAD7F7")   # light purple — target formula cells
F_LGREY  = fill("F2F3F4")   # light grey  — structural input
F_WHITE  = fill("FFFFFF")

def style_hdr(cell, text, bg=F_NAVY, sz=9):
    cell.value = text
    cell.font = Font(bold=True, color="FFFFFF", name="Arial", size=sz)
    cell.fill = bg
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border = thin()

def style_inp(cell, value="", note=""):
    cell.value = value
    cell.font = Font(color="00008B", name="Arial", size=9)
    cell.fill = F_INP
    cell.alignment = Alignment(horizontal="left", vertical="center")
    cell.border = thin()

def style_formula(cell, formula, bg=F_FORM, color="000000"):
    cell.value = formula
    cell.font = Font(color=color, name="Arial", size=9)
    cell.fill = bg
    cell.alignment = Alignment(horizontal="left", vertical="center")
    cell.border = thin()

def style_label(cell, text, bold=False, bg=F_LGREY):
    cell.value = text
    cell.font = Font(bold=bold, name="Arial", size=9, color="1F4E79")
    cell.fill = bg
    cell.alignment = Alignment(horizontal="left", vertical="center")
    cell.border = thin()

def thin():
    s = Side(style="thin", color="CCCCCC")
    return Border(left=s, right=s, top=s, bottom=s)

def set_w(ws, col, w):
    ws.column_dimensions[get_column_letter(col)].width = w

def freeze(ws, cell):
    ws.freeze_panes = cell

def no_grid(ws):
    ws.sheet_view.showGridLines = False

# Climate variable definitions
VARS     = ['rainfall_mm','temperature_celsius','relative_humidity_percent',
            'soil_moisture_percent','wind_speed_10m','dtr_celsius','vpd_kpa','net_solar_rad_kwh_m2']
UNITS    = ['mm','°C','%','%','m/s','°C','kPa','kWh/m²']
KEY_VARS = ['temperature_celsius','vpd_kpa','soil_moisture_percent','dtr_celsius','rainfall_mm']
INTERACTIONS = [(KEY_VARS[i],KEY_VARS[j])
                for i in range(len(KEY_VARS)) for j in range(i+1,len(KEY_VARS))]

# Short labels for interactions
def short(v): 
    return (v.replace("temperature_celsius","temp")
             .replace("relative_humidity_percent","rh")
             .replace("soil_moisture_percent","soil")
             .replace("wind_speed_10m","wind")
             .replace("dtr_celsius","dtr")
             .replace("net_solar_rad_kwh_m2","solar")
             .replace("_kpa","").replace("_mm",""))

NV = len(VARS)          # 8
NI = len(INTERACTIONS)  # 10

# ============================================================
# SHEET 0 — LEGEND / INSTRUCTIONS
# ============================================================
ws0 = wb.active; ws0.title = "LEGEND"; no_grid(ws0)

def legend_row(ws, r, sym, desc, bg, fc="FFFFFF"):
    ws.merge_cells(f"B{r}:B{r}")
    c = ws.cell(r,1); c.value=sym; c.font=Font(name="Arial",size=10,bold=True); c.fill=bg; c.alignment=Alignment(horizontal="center")
    c = ws.cell(r,2); c.value=desc; c.font=Font(name="Arial",size=9); c.fill=F_WHITE
    ws.row_dimensions[r].height=18

ws0.merge_cells("A1:E1")
c=ws0["A1"]; c.value="☕  COFFEE YIELD PREDICTION — EXCEL PIPELINE"
c.font=Font(bold=True,name="Arial",size=14,color="1F4E79"); c.fill=F_INP
c.alignment=Alignment(horizontal="center",vertical="center"); ws0.row_dimensions[1].height=38

ws0.cell(3,1).value="COLOR & CELL LEGEND"
ws0.cell(3,1).font=Font(bold=True,name="Arial",size=11,color="FFFFFF"); ws0.cell(3,1).fill=F_NAVY

entries=[
    ("🔵 LIGHT BLUE cells",  "USER INPUT — only edit these (date, location, tm_ha, produksi_kg, 8 climate vars)", F_INP),
    ("🟢 LIGHT GREEN cells", "FORMULA — annual climate averages, yield, training features",                      F_FORM),
    ("🟡 LIGHT YELLOW cells","FORMULA — quarterly climate averages & features",                                 F_QFORM),
    ("🟠 LIGHT ORANGE cells","FORMULA — pairwise interaction terms (temp×vpd, etc.)",                          F_IFORM),
    ("🟣 LIGHT PURPLE cells","FORMULA — model target: log1p(yield) and inference output",                      F_PFORM),
    ("⬜ LIGHT GREY cells",  "STRUCTURAL INPUT — enter row keys (year, location) to activate formula rows",    F_LGREY),
]
for i,(sym,desc,bg) in enumerate(entries,start=4):
    legend_row(ws0,i,sym,desc,bg)

ws0.cell(11,1).value="WORKFLOW — STEP BY STEP"
ws0.cell(11,1).font=Font(bold=True,name="Arial",size=11,color="FFFFFF"); ws0.cell(11,1).fill=F_NAVY

steps=[
    ("Step 1","RAW_CLIMATE","Enter daily climate data: date, location, 8 variables. Drag row-2 formulas (year/month/quarter) down for all rows."),
    ("Step 2","RAW_PRODUCTION","Enter annual data per village: year, village, tm_ha, produksi_kg."),
    ("Step 3","PRODUCTION_CALC","Drag all formulas from row 2 down to match number of rows in RAW_PRODUCTION. All calculations auto-fill."),
    ("Step 4","ANNUAL_CLIMATE","For each unique (year, location) pair you have in RAW_CLIMATE, enter year+location in cols A+B. Drag row-2 formulas down."),
    ("Step 5","QUARTERLY_CLIMATE","For each unique (year, location, quarter) combo, enter year+location+quarter (1-4). Drag row-2 formulas down."),
    ("Step 6","CLIMATE_FEATURES","For each unique (year, location) pair, enter climate_year+location. All 50+ features auto-calculate."),
    ("Step 7","PREV_YIELD","For each (prod_year, village) in your training data, enter those in A+B. Prev-year yield auto-looks-up."),
    ("Step 8","TRAINING_TABLE","Enter prod_year + village for each training row. All 50 features + yield + prev_yield auto-pull in."),
    ("Step 9","VALIDATION_TABLE","Same as TRAINING_TABLE but for validation/test year rows (e.g. prod_year=2025)."),
    ("Step 10","INFERENCE","Enter new climate values + prev_yield to predict a new harvest. Paste model log output to get kg/ha."),
]
for i,(step,sheet,desc) in enumerate(steps,start=12):
    ws0.cell(i,1).value=step; ws0.cell(i,1).font=Font(bold=True,name="Arial",size=9,color="FFFFFF"); ws0.cell(i,1).fill=F_TEAL
    ws0.cell(i,2).value=f"[{sheet}]"; ws0.cell(i,2).font=Font(bold=True,name="Arial",size=9,color="0E6655")
    ws0.merge_cells(f"C{i}:E{i}")
    ws0.cell(i,3).value=desc; ws0.cell(i,3).font=Font(name="Arial",size=9)
    ws0.row_dimensions[i].height=28

ws0.cell(23,1).value="KEY FORMULA REFERENCE"
ws0.cell(23,1).font=Font(bold=True,name="Arial",size=11,color="FFFFFF"); ws0.cell(23,1).fill=F_NAVY

formulas_ref=[
    ("yield_kg_ha","produksi_kg / tm_ha",    "Basic yield density"),
    ("outlier filter","yield_kg_ha >= 100",     "Villages with < 100 kg/ha are excluded (abandoned/outlier)"),
    ("log_yield (target)","LN(1 + yield_kg_ha)", "Log-transform stabilises variance; model trains on this"),
    ("annual avg","AVERAGEIFS(climate_col, year_col, year, loc_col, location)", "Mean of all daily values for one year per village"),
    ("quarterly avg","AVERAGEIFS(climate_col, year_col, year, loc_col, loc, qtr_col, 1..4)", "Mean of daily values per quarter (Q1-Q4)"),
    ("interaction term","var1_annual × var2_annual", "Pairwise multiplication of annual means for KEY_VARS"),
    ("lag rule","climate year N  →  production year N+1", "Climate of year N predicts the harvest collected in year N+1"),
    ("prev_yield","yield of village in prod_year - 1", "Previous harvest yield as an autoregressive feature"),
    ("inference output","EXP(log_pred) - 1", "Reverses the log transform to get kg/ha back"),
]
for i,(name,formula,note) in enumerate(formulas_ref,start=24):
    ws0.cell(i,1).value=name; ws0.cell(i,1).font=Font(bold=True,name="Arial",size=9)
    ws0.cell(i,2).value=formula; ws0.cell(i,2).font=Font(name="Courier New",size=9,color="0E6655")
    ws0.merge_cells(f"C{i}:E{i}")
    ws0.cell(i,3).value=note; ws0.cell(i,3).font=Font(name="Arial",size=9,italic=True,color="7F8C8D")
    ws0.row_dimensions[i].height=18

for c,w in [(1,18),(2,42),(3,60)]: set_w(ws0,c,w)

# ============================================================
# SHEET 1 — RAW_CLIMATE  (USER INPUT)
# ============================================================
ws1=wb.create_sheet("RAW_CLIMATE"); no_grid(ws1); freeze(ws1,"E2")

# Matches actual CSV column order:
# A=date  B=location  C=latitude(input)  D=longitude(input)
# E=rainfall_mm  F=temperature_celsius  G=rh%  H=soil%  I=wind  J=dtr  K=vpd  L=solar
# M=year[formula]  N=month[formula]  O=quarter[formula]
heads1 = [
    ("date\n(YYYY-MM-DD)","INPUT"),
    ("location\n(village name)","INPUT"),
    ("latitude","INPUT"),
    ("longitude","INPUT"),
]+ [(f"{v}\n({u})","INPUT") for v,u in zip(VARS,UNITS)] + [
    ("year\n[=YEAR(A)]\nDO NOT EDIT","AUTO"),
    ("month\n[=MONTH(A)]\nDO NOT EDIT","AUTO"),
    ("quarter\n[=INT((MONTH-1)/3)+1]\nDO NOT EDIT","AUTO"),
]

bg_map={"INPUT":F_NAVY,"AUTO":F_TEAL}
for ci,(h,t) in enumerate(heads1,1):
    style_hdr(ws1.cell(1,ci),h,bg=bg_map[t])
ws1.row_dimensions[1].height=50

# Sample data rows matching actual CSV structure
samples1=[
    ("2020-01-01","Alam Jaya",4.799346,96.745384, 0.4064,20.2716,83.2215,43.9280,1.0890,9.4165,0.3995,4.6211),
    ("2020-01-02","Alam Jaya",4.799346,96.745384, 0.4071,20.7645,82.3942,42.5227,0.9297,9.9272,0.4321,5.0378),
    ("2020-01-03","Alam Jaya",4.799346,96.745384, 1.3470,21.0985,86.2310,41.9416,0.8083,7.8302,0.3450,4.1395),
    ("2020-04-15","buntul",   4.82,    96.75,     5.2000,20.1000,84.0000,43.5000,1.2000,9.0000,0.4000,4.5000),
    ("2020-07-10","timang gajah",4.78, 96.73,     3.1000,19.8000,85.0000,46.0000,1.3000,8.8000,0.3800,4.4000),
]
for ri,(dt,loc,lat,lon,*cvars) in enumerate(samples1,2):
    style_inp(ws1.cell(ri,1), dt)
    style_inp(ws1.cell(ri,2), loc)
    style_inp(ws1.cell(ri,3), lat)
    style_inp(ws1.cell(ri,4), lon)
    for ci,v in enumerate(cvars,5):            # E=col5 .. L=col12
        style_inp(ws1.cell(ri,ci), round(v,4))
    # M=year, N=month, O=quarter  (cols 13,14,15)
    style_formula(ws1.cell(ri,13), f"=IFERROR(YEAR(A{ri}),\"\")")
    style_formula(ws1.cell(ri,14), f"=IFERROR(MONTH(A{ri}),\"\")")
    style_formula(ws1.cell(ri,15), f"=IFERROR(INT((MONTH(A{ri})-1)/3)+1,\"\")")
    ws1.row_dimensions[ri].height=16

widths1=[22,20,12,12, 14,18,20,18,14,12,10,16, 8,8,10]
for c,w in enumerate(widths1,1): set_w(ws1,c,w)

# drag note
r_note=len(samples1)+3
ws1.merge_cells(f"A{r_note}:O{r_note}")
c=ws1[f"A{r_note}"]
c.value=("← Paste your CSV directly here: A=date, B=location, C=latitude, D=longitude, E-L=8 climate vars. "
         "Drag row-2 formulas in cols M/N/O (year/month/quarter) down for ALL data rows — these feed AVERAGEIFS in other sheets.")
c.font=Font(italic=True,color="1F4E79",name="Arial",size=9)

# ============================================================
# SHEET 2 — RAW_PRODUCTION  (USER INPUT)
# ============================================================
ws2=wb.create_sheet("RAW_PRODUCTION"); no_grid(ws2); freeze(ws2,"A2")

heads2=[("year","INPUT"),("village\n(lowercase)","INPUT"),
        ("tm_ha\n(bearing area, ha)","INPUT"),("produksi_kg\n(total production, kg)","INPUT")]
for ci,(h,t) in enumerate(heads2,1):
    style_hdr(ws2.cell(1,ci),h,bg=F_NAVY)
ws2.row_dimensions[1].height=40

prod_samples=[
    (2021,"buntul",120.5,95000),
    (2021,"timang gajah",85.0,72000),
    (2021,"wih pesam",60.0,51000),
    (2022,"buntul",122.0,98000),
    (2022,"timang gajah",86.5,74500),
    (2022,"wih pesam",61.0,52000),
    (2023,"buntul",125.0,50000),
]
for ri,(yr,vill,tm,prod) in enumerate(prod_samples,2):
    for ci,v in enumerate([yr,vill,tm,prod],1):
        style_inp(ws2.cell(ri,ci),v)
    ws2.row_dimensions[ri].height=16

for c,w in [(1,8),(2,20),(3,18),(4,20)]: set_w(ws2,c,w)
ws2.merge_cells(f"A{len(prod_samples)+3}:D{len(prod_samples)+3}")
ws2[f"A{len(prod_samples)+3}"].value="← Add your rows here. Keep village names lowercase & consistent with RAW_CLIMATE location."
ws2[f"A{len(prod_samples)+3}"].font=Font(italic=True,color="1F4E79",name="Arial",size=9)

# ============================================================
# SHEET 3 — PRODUCTION_CALC  (all formulas from RAW_PRODUCTION)
# ============================================================
ws3=wb.create_sheet("PRODUCTION_CALC"); no_grid(ws3); freeze(ws3,"A2")

heads3=[
    ("row#\n(from RAW_PRODUCTION)","AUTO"),
    ("year","AUTO"),("village\n(normalized)","AUTO"),("tm_ha","AUTO"),("produksi_kg","AUTO"),
    ("key\n= year_village","AUTO"),
    ("tm_valid?\n(tm_ha > 0)","AUTO"),
    ("yield_kg_ha_RAW\n= produksi_kg / tm_ha","AUTO"),
    ("outlier_flag\n(< 100 → OUTLIER)","AUTO"),
    ("yield_kg_ha_FILTERED\n(only if ≥ 100 kg/ha)","AUTO"),
    ("log1p_yield\n= LN(1 + yield_filtered)\n[MODEL TARGET]","AUTO"),
]
bgs3=[F_DKGREY]+[F_TEAL]*4+[F_TEAL]+[F_TEAL]+[F_TEAL]+[F_TEAL]+[F_TEAL]+[F_PURPLE]
for ci,(h,t) in enumerate(heads3,1):
    style_hdr(ws3.cell(1,ci),h,bg=bgs3[ci-1])
ws3.row_dimensions[1].height=55

for ri in range(2, len(prod_samples)+2):
    src=ri   # same row number in RAW_PRODUCTION
    style_formula(ws3.cell(ri,1),  f"={src}-1",                                              bg=F_LGREY)  # row#
    style_formula(ws3.cell(ri,2),  f"=RAW_PRODUCTION!A{src}")                                             # year
    style_formula(ws3.cell(ri,3),  f'=IFERROR(LOWER(TRIM(RAW_PRODUCTION!B{src})),"")')                    # village norm
    style_formula(ws3.cell(ri,4),  f"=RAW_PRODUCTION!C{src}")                                             # tm_ha
    style_formula(ws3.cell(ri,5),  f"=RAW_PRODUCTION!D{src}")                                             # produksi_kg
    style_formula(ws3.cell(ri,6),  f'=IFERROR(B{ri}&"_"&C{ri},"")')                                      # key
    style_formula(ws3.cell(ri,7),  f'=IF(D{ri}>0,"YES","NO — skip (tm_ha ≤ 0)")')                        # tm_valid
    style_formula(ws3.cell(ri,8),  f"=IFERROR(IF(D{ri}>0,E{ri}/D{ri},\"\"),\"\")")                       # yield raw
    style_formula(ws3.cell(ri,9),  f'=IFERROR(IF(ISNUMBER(H{ri}),IF(H{ri}<100,"OUTLIER (< 100 kg/ha)","OK"),""),"")') # flag
    style_formula(ws3.cell(ri,10), f'=IFERROR(IF(AND(ISNUMBER(H{ri}),H{ri}>=100),H{ri},""),"")') # filtered
    style_formula(ws3.cell(ri,11), f'=IFERROR(IF(ISNUMBER(J{ri}),LN(1+J{ri}),""),"")' ,bg=F_PFORM) # log target
    ws3.row_dimensions[ri].height=16

widths3=[8,8,20,12,16,22,22,20,28,22,20]
for c,w in enumerate(widths3,1): set_w(ws3,c,w)

drag_row3=len(prod_samples)+3
ws3.merge_cells(f"A{drag_row3}:K{drag_row3}")
c=ws3[f"A{drag_row3}"]
c.value="← Drag ALL formulas in row 2 (cols A-K) down to match the number of rows in RAW_PRODUCTION."
c.font=Font(italic=True,color="1F4E79",name="Arial",size=9)

# ============================================================
# SHEET 4 — ANNUAL_CLIMATE
# A=year(key-input) B=location(key-input) C=key D..K=8 annual averages
# ============================================================
ws4=wb.create_sheet("ANNUAL_CLIMATE"); no_grid(ws4); freeze(ws4,"D2")

# RAW_CLIMATE column letters for the 8 vars: F G H I J K L M (cols 6-13)
RC_VAR_COLS = [get_column_letter(6+i) for i in range(8)]  # F,G,H,I,J,K,L,M

heads4=[("climate_year\n[structural input]","KEY"),("location\n[structural input]","KEY"),
        ("key\n= year_location","AUTO")]
heads4 += [(f"{v}\n(annual mean)\n[=AVERAGEIFS(...)]","AUTO") for v in VARS]

bgs4=[F_DKGREY,F_DKGREY,F_TEAL]+[F_TEAL]*8
for ci,(h,t) in enumerate(heads4,1):
    style_hdr(ws4.cell(1,ci),h,bg=bgs4[ci-1])
ws4.row_dimensions[1].height=55

# Build unique year-location pairs from samples
import itertools
ann_pairs=[(2020,"buntul"),(2020,"timang gajah"),(2020,"wih pesam"),
           (2021,"buntul"),(2021,"timang gajah"),(2021,"wih pesam"),
           (2022,"buntul"),(2022,"timang gajah"),(2022,"wih pesam"),
           (2023,"buntul"),(2023,"timang gajah"),(2023,"wih pesam")]

for ri,(yr,loc) in enumerate(ann_pairs,2):
    style_label(ws4.cell(ri,1), yr)
    style_label(ws4.cell(ri,2), loc)
    style_formula(ws4.cell(ri,3), f'=IFERROR(A{ri}&"_"&B{ri},"")')
    for vi,rcol in enumerate(RC_VAR_COLS):
        coli = 4+vi
        f=(f"=IFERROR(AVERAGEIFS("
           f"RAW_CLIMATE!${rcol}:${rcol},"
           f"RAW_CLIMATE!$C:$C,$A{ri},"
           f"RAW_CLIMATE!$B:$B,$B{ri}),\"\")")
        style_formula(ws4.cell(ri,coli), f)
    ws4.row_dimensions[ri].height=16

widths4=[12,20,26]+[18]*8
for c,w in enumerate(widths4,1): set_w(ws4,c,w)

drag_row4=len(ann_pairs)+3
ws4.merge_cells(f"A{drag_row4}:K{drag_row4}")
c=ws4[f"A{drag_row4}"]
c.value=("← Enter each unique (year, location) pair in cols A+B. "
         "Drag row-2 formulas (C to K) down for all pairs. "
         "Pairs must match exactly what's in RAW_CLIMATE column B.")
c.font=Font(italic=True,color="1F4E79",name="Arial",size=9)

# ============================================================
# SHEET 5 — QUARTERLY_CLIMATE
# A=year B=location C=quarter(1-4) D=key E..L=8 quarterly averages
# ============================================================
ws5=wb.create_sheet("QUARTERLY_CLIMATE"); no_grid(ws5); freeze(ws5,"E2")

heads5=[("climate_year\n[structural input]","KEY"),("location\n[structural input]","KEY"),
        ("quarter\n(1/2/3/4)\n[structural input]","KEY"),
        ("key\n= year_location_quarter","AUTO")]
heads5 += [(f"{v}\n(quarterly mean)\n[=AVERAGEIFS(..., quarter)]","AUTO") for v in VARS]

bgs5=[F_DKGREY]*3+[F_TEAL]+[F_TEAL]*8
for ci,(h,t) in enumerate(heads5,1):
    style_hdr(ws5.cell(1,ci),h,bg=bgs5[ci-1],sz=8)
ws5.row_dimensions[1].height=55

# One row per (year, location, quarter)
qrt_rows=[]
for (yr,loc) in ann_pairs:
    for q in range(1,5):
        qrt_rows.append((yr,loc,q))

for ri,(yr,loc,q) in enumerate(qrt_rows,2):
    style_label(ws5.cell(ri,1), yr)
    style_label(ws5.cell(ri,2), loc)
    style_label(ws5.cell(ri,3), q)
    style_formula(ws5.cell(ri,4), f'=IFERROR(A{ri}&"_"&B{ri}&"_"&C{ri},"")')
    for vi,rcol in enumerate(RC_VAR_COLS):
        coli=5+vi
        # RAW_CLIMATE: B=location, C=year, E=quarter, var=rcol
        f=(f"=IFERROR(AVERAGEIFS("
           f"RAW_CLIMATE!${rcol}:${rcol},"
           f"RAW_CLIMATE!$C:$C,$A{ri},"
           f"RAW_CLIMATE!$B:$B,$B{ri},"
           f"RAW_CLIMATE!$E:$E,$C{ri}),\"\")")
        style_formula(ws5.cell(ri,coli), f, bg=F_QFORM)
    ws5.row_dimensions[ri].height=16

widths5=[12,20,10,30]+[18]*8
for c,w in enumerate(widths5,1): set_w(ws5,c,w)

drag_row5=len(qrt_rows)+3
ws5.merge_cells(f"A{drag_row5}:L{drag_row5}")
c=ws5[f"A{drag_row5}"]
c.value=("← Enter each unique (year, location, quarter) triple in cols A+B+C. "
         "For each year+location you have 4 rows (Q=1,2,3,4). "
         "Drag row-2 formulas (D to L) down for all rows.")
c.font=Font(italic=True,color="1F4E79",name="Arial",size=9)

# ============================================================
# SHEET 6 — CLIMATE_FEATURES
# Assembles all 50 features + prod_year per (climate_year, location)
#
# Col layout:
#  A  = climate_year (key input)
#  B  = location     (key input)
#  C  = key          (formula)
#  D..K   = 8 annual means      (VLOOKUP ANNUAL_CLIMATE !$C:$K, offsets 2-9)
#  L..S   = 8 Q1 vars           (VLOOKUP QUARTERLY !$D:$L, offsets 2-9, key=yr_loc_1)
#  T..AA  = 8 Q2 vars           (... key=yr_loc_2)
#  AB..AI = 8 Q3 vars           (... key=yr_loc_3)
#  AJ..AQ = 8 Q4 vars           (... key=yr_loc_4)
#  AR..BA = 10 interactions      (formula from annual cols D..K)
#  BB     = prod_year            (= climate_year + 1)
# ============================================================
ws6=wb.create_sheet("CLIMATE_FEATURES"); no_grid(ws6); freeze(ws6,"D2")

# --- Build headers ---
CF_HEADS=[]
CF_HEADS.append(("climate_year\n[structural input]",F_DKGREY))
CF_HEADS.append(("location\n[structural input]",F_DKGREY))
CF_HEADS.append(("key\n= climate_year_location",F_TEAL))

for v in VARS:
    CF_HEADS.append((f"{v}\nannual avg",F_NAVY))
for q in range(1,5):
    for v in VARS:
        CF_HEADS.append((f"{v}\nQ{q}",F_TEAL))
for v1,v2 in INTERACTIONS:
    CF_HEADS.append((f"{short(v1)}\n×\n{short(v2)}",F_ORANGE))
CF_HEADS.append(("prod_year\n= climate_year + 1\n[LAG]",F_PURPLE))

for ci,(h,bg) in enumerate(CF_HEADS,1):
    style_hdr(ws6.cell(1,ci),h,bg=bg,sz=8)
ws6.row_dimensions[1].height=50

# --- Column index helpers ---
# C=3 → key; D=4 → annual start; L=12 → Q1 start; T=20 → Q2; AB=28 → Q3; AJ=36 → Q4
# AR=44 → interactions; BB=54 → prod_year
ANN_START = 4   # col D
Q_STARTS  = [12, 20, 28, 36]  # L, T, AB, AJ
ITR_START = 44  # AR
PROD_YEAR_COL = 54  # BB

# Annual col letters for 8 vars (D..K)
ann_col_letters = [get_column_letter(ANN_START+i) for i in range(8)]  # D,E,F,G,H,I,J,K
# VARS index mapping for KEY_VARS in annual cols:
kv_annual_col = {v: ann_col_letters[VARS.index(v)] for v in KEY_VARS}
# temperature=E(4+1=5→E), vpd=J(4+6=10→J), soil=G(4+3=7→G), dtr=I(4+5=9→I), rainfall=D(4+0=4→D)

# ANNUAL_CLIMATE: key in col C, vars in D..K → VLOOKUP range $C:$K, offsets 2..9
# QUARTERLY_CLIMATE: key in col D, vars in E..L → VLOOKUP range $D:$L, offsets 2..9

cf_pairs = ann_pairs   # same year-location pairs

for ri,(yr,loc) in enumerate(cf_pairs,2):
    # A: climate_year
    style_label(ws6.cell(ri,1), yr)
    # B: location
    style_label(ws6.cell(ri,2), loc)
    # C: key
    style_formula(ws6.cell(ri,3), f'=IFERROR(A{ri}&"_"&B{ri},"")')

    # D..K: 8 annual means from ANNUAL_CLIMATE
    for vi in range(8):
        ci=ANN_START+vi
        off=vi+2   # VLOOKUP offset from key col C of ANNUAL_CLIMATE
        style_formula(ws6.cell(ri,ci),
            f'=IFERROR(VLOOKUP($C{ri},ANNUAL_CLIMATE!$C:$K,{off},0),"")')

    # Q1..Q4: 8 vars each from QUARTERLY_CLIMATE
    for q in range(1,5):
        qs = Q_STARTS[q-1]
        for vi in range(8):
            ci=qs+vi
            off=vi+2   # offset in QUARTERLY_CLIMATE $D:$L
            qkey = f'$A{ri}&"_"&$B{ri}&"_{q}"'
            style_formula(ws6.cell(ri,ci),
                f'=IFERROR(VLOOKUP({qkey},QUARTERLY_CLIMATE!$D:$L,{off},0),"")',
                bg=F_QFORM)

    # Interactions: multiply annual-mean cols for the 5 KEY_VARS
    for ii,(v1,v2) in enumerate(INTERACTIONS):
        ci=ITR_START+ii
        c1=f"{kv_annual_col[v1]}{ri}"
        c2=f"{kv_annual_col[v2]}{ri}"
        style_formula(ws6.cell(ri,ci),
            f'=IFERROR(IF(AND(ISNUMBER({c1}),ISNUMBER({c2})),{c1}*{c2},""),"")' ,
            bg=F_IFORM)

    # prod_year
    style_formula(ws6.cell(ri,PROD_YEAR_COL),
        f"=IFERROR(A{ri}+1,\"\")", bg=F_PFORM)

    ws6.row_dimensions[ri].height=16

# column widths
for ci in range(1,PROD_YEAR_COL+1):
    set_w(ws6,ci,16)
set_w(ws6,1,14); set_w(ws6,2,18); set_w(ws6,3,26)

drag_row6=len(cf_pairs)+3
ws6.merge_cells(f"A{drag_row6}:D{drag_row6}")
c=ws6[f"A{drag_row6}"]
c.value=("← Enter (climate_year, location) in cols A+B — one row per unique pair in your data. "
         "Drag row-2 formulas across all cols and down for all rows. "
         "Quarterly values auto-pull from QUARTERLY_CLIMATE.")
c.font=Font(italic=True,color="1F4E79",name="Arial",size=9)

# ============================================================
# SHEET 7 — PREV_YIELD
# For each (prod_year, village) → looks up yield from prod_year-1
# ============================================================
ws7=wb.create_sheet("PREV_YIELD"); no_grid(ws7); freeze(ws7,"A2")

heads7=[
    ("prod_year\n[structural input]",F_DKGREY),
    ("village\n[structural input]",F_DKGREY),
    ("key\n= prod_year_village",F_TEAL),
    ("prev_prod_year\n= prod_year - 1",F_TEAL),
    ("prev_lookup_key\n= (prod_year-1)_village",F_TEAL),
    ("prev_yield_direct\n[INDEX-MATCH in\nPRODUCTION_CALC]",F_TEAL),
    ("village_avg_yield\n[fallback if no\nprev year data]",F_TEAL),
    ("prev_yield_FINAL\n[direct if found,\nelse village avg]",F_FORM),
]
for ci,(h,bg) in enumerate(heads7,1):
    style_hdr(ws7.cell(1,ci),h,bg=bg,sz=8)
ws7.row_dimensions[1].height=55

# Build list: all (prod_year, village) combos needed for training
# prod_year = production year, which maps from climate year + 1
# For training: prod_years 2021..2024 (since climate data is 2020-2024 → yields 2021-2025)
pv_rows=[]
villages=["buntul","timang gajah","wih pesam"]
for yr in [2021,2022,2023,2024]:
    for v in villages:
        pv_rows.append((yr,v))

for ri,(yr,vill) in enumerate(pv_rows,2):
    style_label(ws7.cell(ri,1), yr)
    style_label(ws7.cell(ri,2), vill)
    # C: key
    style_formula(ws7.cell(ri,3), f'=IFERROR(A{ri}&"_"&B{ri},"")')
    # D: prev_prod_year = A-1
    style_formula(ws7.cell(ri,4), f'=IFERROR(A{ri}-1,"")')
    # E: prev lookup key
    style_formula(ws7.cell(ri,5), f'=IFERROR((A{ri}-1)&"_"&B{ri},"")')
    # F: direct lookup in PRODUCTION_CALC col F=key(6), col J=yield_filtered(10)
    style_formula(ws7.cell(ri,6),
        f'=IFERROR(INDEX(PRODUCTION_CALC!$J:$J,MATCH($E{ri},PRODUCTION_CALC!$F:$F,0)),"")')
    # G: village average yield as fallback (AVERAGEIF on PRODUCTION_CALC col C=village, col J=yield)
    style_formula(ws7.cell(ri,7),
        f'=IFERROR(AVERAGEIF(PRODUCTION_CALC!$C:$C,$B{ri},PRODUCTION_CALC!$J:$J),"")')
    # H: final prev_yield
    style_formula(ws7.cell(ri,8),
        f'=IFERROR(IF(ISNUMBER(F{ri}),F{ri},IF(ISNUMBER(G{ri}),G{ri},"")),"")' ,
        bg=F_FORM)
    ws7.row_dimensions[ri].height=16

widths7=[12,20,22,16,26,22,22,22]
for c,w in enumerate(widths7,1): set_w(ws7,c,w)

drag_row7=len(pv_rows)+3
ws7.merge_cells(f"A{drag_row7}:H{drag_row7}")
c=ws7[f"A{drag_row7}"]
c.value=("← Enter each (prod_year, village) pair you need. "
         "Col F looks up the actual prev-year yield. Col G is the village average as fallback. "
         "Col H picks direct if available, else average. Drag row-2 formulas down.")
c.font=Font(italic=True,color="1F4E79",name="Arial",size=9)

# ============================================================
# SHEET 8 — TRAINING_TABLE  (the full feature matrix)
#
# Col layout:
#  A  = prod_year           (structural input)
#  B  = village             (structural input)
#  C  = climate_year        = prod_year - 1
#  D  = climate_key         = climate_year_village  (lookup key for CLIMATE_FEATURES)
#  E  = prod_key            = prod_year_village     (lookup key for PRODUCTION_CALC + PREV_YIELD)
#  F..M   = 8 annual features  (VLOOKUP CLIMATE_FEATURES, key col=C(3), offsets 2..9)
#  N..U   = 8 Q1 features      (offsets 10..17)
#  V..AC  = 8 Q2 features      (offsets 18..25)
#  AD..AK = 8 Q3 features      (offsets 26..33)
#  AL..AS = 8 Q4 features      (offsets 34..41)
#  AT..BC = 10 interactions    (offsets 42..51)
#  BD     = prev_yield_kg_ha  (VLOOKUP PREV_YIELD key col=C, offset=6)
#  BE     = yield_kg_ha       (INDEX-MATCH PRODUCTION_CALC col F→J)
#  BF     = log1p_yield       (INDEX-MATCH PRODUCTION_CALC col F→K) = LN(1+yield)
# ============================================================
ws8=wb.create_sheet("TRAINING_TABLE"); no_grid(ws8); freeze(ws8,"F2")

# CLIMATE_FEATURES key col = C (pos 3), last col = BB (pos 54)
# So VLOOKUP range = $C:$BB
# Offsets from key col C (1-indexed): annual D=2..K=9, Q1 L=10..S=17, Q2 T=18..AA=25,
#  Q3 AB=26..AI=33, Q4 AJ=34..AQ=41, interactions AR=42..BA=51, prod_year BB=52
CF_LAST = get_column_letter(PROD_YEAR_COL)   # BB

TT_HEADS=[]
TT_HEADS.append(("prod_year\n[structural input]",F_DKGREY))
TT_HEADS.append(("village\n[structural input]",F_DKGREY))
TT_HEADS.append(("climate_year\n= prod_year - 1",F_TEAL))
TT_HEADS.append(("climate_key\n= climate_year_village\n[lookup→CLIMATE_FEATURES]",F_TEAL))
TT_HEADS.append(("prod_key\n= prod_year_village\n[lookup→PRODUCTION_CALC]",F_TEAL))

for v in VARS:
    TT_HEADS.append((f"[feat] {v}\nannual avg",F_NAVY))
for q in range(1,5):
    for v in VARS:
        TT_HEADS.append((f"[feat] {v}\nQ{q}",F_TEAL))
for v1,v2 in INTERACTIONS:
    TT_HEADS.append((f"[feat] {short(v1)}\n×\n{short(v2)}",F_ORANGE))
TT_HEADS.append(("prev_yield_kg_ha\n[feat from PREV_YIELD]",F_TEAL))
TT_HEADS.append(("yield_kg_ha\n[target raw — from\nPRODUCTION_CALC]",F_PURPLE))
TT_HEADS.append(("log1p_yield\n= LN(1 + yield)\n[MODEL TARGET]",F_PURPLE))

for ci,(h,bg) in enumerate(TT_HEADS,1):
    style_hdr(ws8.cell(1,ci),h,bg=bg,sz=8)
ws8.row_dimensions[1].height=55

# Feature column offsets in CLIMATE_FEATURES (from key col C=3):
# annual D(4)→off2, E(5)→off3, ..., K(11)→off9
# Q1: L(12)→off10, ..., S(19)→off17
# Q2: T(20)→off18, ..., AA(27)→off25
# Q3: AB(28)→off26, ..., AI(35)→off33
# Q4: AJ(36)→off34, ..., AQ(43)→off41
# interactions: AR(44)→off42, ..., BA(53)→off51
def cf_offset(col_1based):
    return col_1based - 3 + 1  # relative to col C key

# Training rows
tt_rows=pv_rows  # same (prod_year, village) combos

for ri,(yr,vill) in enumerate(tt_rows,2):
    col=1
    # A: prod_year
    style_label(ws8.cell(ri,col), yr); col+=1
    # B: village
    style_label(ws8.cell(ri,col), vill); col+=1
    # C: climate_year
    style_formula(ws8.cell(ri,col), f"=IFERROR(A{ri}-1,\"\")"); col+=1
    # D: climate_key
    style_formula(ws8.cell(ri,col), f'=IFERROR((A{ri}-1)&"_"&B{ri},"")'); col+=1
    # E: prod_key
    style_formula(ws8.cell(ri,col), f'=IFERROR(A{ri}&"_"&B{ri},"")'); col+=1

    # F..M: 8 annual features
    for vi in range(8):
        off = cf_offset(ANN_START + vi)   # ANN_START=4 → off=2..9
        style_formula(ws8.cell(ri,col),
            f'=IFERROR(VLOOKUP($D{ri},CLIMATE_FEATURES!$C:${CF_LAST},{off},0),"")'); col+=1

    # Q1..Q4: 8 vars each
    for q in range(4):
        for vi in range(8):
            off = cf_offset(Q_STARTS[q] + vi)
            style_formula(ws8.cell(ri,col),
                f'=IFERROR(VLOOKUP($D{ri},CLIMATE_FEATURES!$C:${CF_LAST},{off},0),"")' ,
                bg=F_QFORM); col+=1

    # Interactions: 10
    for ii in range(NI):
        off = cf_offset(ITR_START + ii)
        style_formula(ws8.cell(ri,col),
            f'=IFERROR(VLOOKUP($D{ri},CLIMATE_FEATURES!$C:${CF_LAST},{off},0),"")' ,
            bg=F_IFORM); col+=1

    # BD: prev_yield — VLOOKUP PREV_YIELD (key=col C, prev_yield_FINAL=col H → off=6)
    style_formula(ws8.cell(ri,col),
        f'=IFERROR(VLOOKUP($E{ri},PREV_YIELD!$C:$H,6,0),"")' , bg=F_QFORM); col+=1

    # BE: yield_kg_ha from PRODUCTION_CALC (key=col F, yield_filtered=col J → INDEX/MATCH)
    style_formula(ws8.cell(ri,col),
        f'=IFERROR(INDEX(PRODUCTION_CALC!$J:$J,MATCH($E{ri},PRODUCTION_CALC!$F:$F,0)),"")' ,
        bg=F_PFORM); col+=1

    # BF: log1p_yield from PRODUCTION_CALC (key=col F, log1p=col K)
    style_formula(ws8.cell(ri,col),
        f'=IFERROR(INDEX(PRODUCTION_CALC!$K:$K,MATCH($E{ri},PRODUCTION_CALC!$F:$F,0)),"")' ,
        bg=F_PFORM); col+=1

    ws8.row_dimensions[ri].height=16

# Col widths
for ci in range(1,len(TT_HEADS)+1):
    set_w(ws8,ci,16)
set_w(ws8,1,12); set_w(ws8,2,18)
set_w(ws8,3,14); set_w(ws8,4,26); set_w(ws8,5,26)

drag_row8=len(tt_rows)+3
ws8.merge_cells(f"A{drag_row8}:F{drag_row8}")
c=ws8[f"A{drag_row8}"]
c.value=("← Enter each (prod_year, village) pair. Drag ALL formulas from row 2 down. "
         "Make sure CLIMATE_FEATURES, PRODUCTION_CALC, and PREV_YIELD are filled first.")
c.font=Font(italic=True,color="1F4E79",name="Arial",size=9)

# ============================================================
# SHEET 9 — VALIDATION_TABLE  (same structure, 2025 prod_year)
# ============================================================
ws9=wb.create_sheet("VALIDATION_TABLE"); no_grid(ws9); freeze(ws9,"F2")

ws9.merge_cells("A1:F1")
c=ws9["A1"]
c.value=("VALIDATION TABLE — same formulas as TRAINING_TABLE. "
         "Enter prod_year=2025 (or whichever hold-out year) + village in cols A+B. "
         "Drag row-3 formulas down for all validation rows.")
c.font=Font(bold=True,name="Arial",size=10,color="FFFFFF"); c.fill=F_PURPLE
c.alignment=Alignment(horizontal="left",vertical="center",wrap_text=True)
ws9.row_dimensions[1].height=30

# Same headers as TRAINING_TABLE
for ci,(h,bg) in enumerate(TT_HEADS,1):
    style_hdr(ws9.cell(2,ci),h,bg=bg,sz=8)
ws9.row_dimensions[2].height=55

# 3 example validation rows (2025 prod_year)
val_rows=[(2025,"buntul"),(2025,"timang gajah"),(2025,"wih pesam")]
for ri,(yr,vill) in enumerate(val_rows,3):
    col=1
    style_label(ws9.cell(ri,col), yr);  col+=1
    style_label(ws9.cell(ri,col), vill); col+=1
    style_formula(ws9.cell(ri,col), f"=IFERROR(A{ri}-1,\"\")"); col+=1
    style_formula(ws9.cell(ri,col), f'=IFERROR((A{ri}-1)&"_"&B{ri},"")'); col+=1
    style_formula(ws9.cell(ri,col), f'=IFERROR(A{ri}&"_"&B{ri},"")'); col+=1

    for vi in range(8):
        off=cf_offset(ANN_START+vi)
        style_formula(ws9.cell(ri,col),
            f'=IFERROR(VLOOKUP($D{ri},CLIMATE_FEATURES!$C:${CF_LAST},{off},0),"")'); col+=1

    for q in range(4):
        for vi in range(8):
            off=cf_offset(Q_STARTS[q]+vi)
            style_formula(ws9.cell(ri,col),
                f'=IFERROR(VLOOKUP($D{ri},CLIMATE_FEATURES!$C:${CF_LAST},{off},0),"")' ,
                bg=F_QFORM); col+=1

    for ii in range(NI):
        off=cf_offset(ITR_START+ii)
        style_formula(ws9.cell(ri,col),
            f'=IFERROR(VLOOKUP($D{ri},CLIMATE_FEATURES!$C:${CF_LAST},{off},0),"")' ,
            bg=F_IFORM); col+=1

    style_formula(ws9.cell(ri,col),
        f'=IFERROR(VLOOKUP($E{ri},PREV_YIELD!$C:$H,6,0),"")' , bg=F_QFORM); col+=1
    style_formula(ws9.cell(ri,col),
        f'=IFERROR(INDEX(PRODUCTION_CALC!$J:$J,MATCH($E{ri},PRODUCTION_CALC!$F:$F,0)),"")' ,
        bg=F_PFORM); col+=1
    style_formula(ws9.cell(ri,col),
        f'=IFERROR(INDEX(PRODUCTION_CALC!$K:$K,MATCH($E{ri},PRODUCTION_CALC!$F:$F,0)),"")' ,
        bg=F_PFORM); col+=1

    ws9.row_dimensions[ri].height=16

for ci in range(1,len(TT_HEADS)+1): set_w(ws9,ci,16)
set_w(ws9,1,12); set_w(ws9,2,18); set_w(ws9,3,14); set_w(ws9,4,26); set_w(ws9,5,26)

# ============================================================
# SHEET 10 — INFERENCE
# Enter new climate scenario → feature vector → paste log_pred → get kg/ha
# ============================================================
ws10=wb.create_sheet("INFERENCE"); no_grid(ws10)

def sec(ws,r,text,bg=F_NAVY,cols="A:F"):
    a,b=cols.split(":")
    ca=ord(a)-64; cb=ord(b)-64
    ws.merge_cells(f"{a}{r}:{b}{r}")
    c=ws[f"{a}{r}"]
    c.value=text; c.font=Font(bold=True,name="Arial",size=10,color="FFFFFF")
    c.fill=bg; c.alignment=Alignment(horizontal="left",vertical="center")
    ws.row_dimensions[r].height=22

# Title
ws10.merge_cells("A1:G1")
c=ws10["A1"]; c.value="☕  INFERENCE — Coffee Yield Prediction for a New Year/Village"
c.font=Font(bold=True,name="Arial",size=13,color="1F4E79"); c.fill=F_INP
c.alignment=Alignment(horizontal="center",vertical="center"); ws10.row_dimensions[1].height=35

# ─── A. Annual climate ───────────────────────────────────────────────────────
sec(ws10, 3, "SECTION A — Enter Annual Average Climate  (climate year N; predicts harvest N+1)")
for ci,h in enumerate(["Variable","Annual Average Value","Unit","Source / Note"],1):
    style_hdr(ws10.cell(4,ci),h,bg=F_TEAL,sz=9)
ws10.row_dimensions[4].height=20

ann_defaults=[1850.0,20.3,83.5,44.0,1.1,9.2,0.41,4.6]
ann_rows_inf={}
for vi,(v,u,d) in enumerate(zip(VARS,UNITS,ann_defaults)):
    r=5+vi
    style_label(ws10.cell(r,1), v)
    style_inp(ws10.cell(r,2), d)
    style_label(ws10.cell(r,3), u, bg=F_WHITE)
    style_label(ws10.cell(r,4), "Mean of all daily values for the climate year", bg=F_WHITE)
    ann_rows_inf[v]=r
    ws10.row_dimensions[r].height=16

# ─── B. Quarterly climate ────────────────────────────────────────────────────
sec(ws10, 14, "SECTION B — Enter Quarterly Average Climate  (same climate year N)", bg=F_TEAL)
ws10.cell(15,1).value="Variable"
ws10.cell(15,1).font=Font(bold=True,name="Arial",size=9)
for qi,lbl in enumerate(["Q1 (Jan-Mar)","Q2 (Apr-Jun)","Q3 (Jul-Sep)","Q4 (Oct-Dec)"],2):
    style_hdr(ws10.cell(15,qi),lbl,bg=F_TEAL,sz=9)
ws10.row_dimensions[15].height=20

q_defs={
    'rainfall_mm':               [420,480,510,440],
    'temperature_celsius':       [19.8,20.1,20.5,20.8],
    'relative_humidity_percent': [84,83,82,85],
    'soil_moisture_percent':     [44,43,45,44],
    'wind_speed_10m':            [1.0,1.1,1.2,1.1],
    'dtr_celsius':               [9.5,9.2,9.0,9.1],
    'vpd_kpa':                   [0.38,0.40,0.43,0.42],
    'net_solar_rad_kwh_m2':      [4.5,4.6,4.8,4.5],
}
q_cells_inf={}  # {(var,q): "B16"}
for vi,v in enumerate(VARS):
    r=16+vi
    style_label(ws10.cell(r,1),v)
    for qi in range(4):
        c_addr=f"{get_column_letter(2+qi)}{r}"
        style_inp(ws10.cell(r,2+qi), q_defs[v][qi])
        q_cells_inf[(v,qi+1)]=c_addr
    ws10.row_dimensions[r].height=16

# ─── C. Interaction terms ────────────────────────────────────────────────────
sec(ws10, 25, "SECTION C — Interaction Terms  (auto-calculated from Section A annual values)", bg=F_ORANGE, cols="A:E")
for ci,h in enumerate(["Interaction","Formula","Calculated Value"],1):
    style_hdr(ws10.cell(26,ci),h,bg=F_ORANGE,sz=9)
ws10.row_dimensions[26].height=18

intr_cells_inf={}
for ii,(v1,v2) in enumerate(INTERACTIONS):
    r=27+ii
    style_label(ws10.cell(r,1), f"{v1}  ×  {v2}")
    r1=ann_rows_inf[v1]; r2=ann_rows_inf[v2]
    style_label(ws10.cell(r,2), f"B{r1}  x  B{r2}", bg=F_WHITE)
    style_formula(ws10.cell(r,3), f"=IFERROR(B{r1}*B{r2},\"\")", bg=F_IFORM)
    intr_cells_inf[(v1,v2)]=f"C{r}"
    ws10.row_dimensions[r].height=16

# ─── D. prev_yield + area ─────────────────────────────────────────────────────
sec(ws10, 38, "SECTION D — Previous Year Yield & Farm Area", bg=F_PURPLE, cols="A:E")
for r,(lbl,val,nt) in enumerate([(
        "prev_yield_kg_ha  (actual yield from harvest year N)",750.0,
        "Enter the village yield (kg/ha) from the previous harvest year"),
    ("luas_lahan_ha  (TM bearing area of this village)",150.0,
        "Bearing area in hectares — used to calculate total production")],start=39):
    style_label(ws10.cell(r,1),lbl,bold=True)
    style_inp(ws10.cell(r,2),val)
    style_label(ws10.cell(r,3),nt,bg=F_WHITE)
    ws10.row_dimensions[r].height=18

# ─── E. Feature vector ──────────────────────────────────────────────────────
sec(ws10, 42, "SECTION E — Complete Feature Vector  (feed to Python model in this exact order)", bg=F_TEAL, cols="A:D")
for ci,h in enumerate(["#","Feature Name","Value  ← feeds to model.predict()","Category"],1):
    style_hdr(ws10.cell(43,ci),h,bg=F_TEAL,sz=9)
ws10.row_dimensions[43].height=20

all_feats = (
    [(v,"annual") for v in VARS] +
    [(f"{v}_Q{q}","quarterly") for q in range(1,5) for v in VARS] +
    [(f"{v1}_x_{v2}","interaction") for v1,v2 in INTERACTIONS] +
    [("prev_yield_kg_ha","yield-lag")]
)
cat_bg={"annual":F_FORM,"quarterly":F_QFORM,"interaction":F_IFORM,"yield-lag":F_QFORM}

for fi,(fname,cat) in enumerate(all_feats):
    r=44+fi
    ws10.cell(r,1).value=fi+1
    ws10.cell(r,1).font=Font(name="Arial",size=8,color="7F8C8D")
    style_label(ws10.cell(r,2), fname)
    ws10.cell(r,2).font=Font(name="Courier New",size=8)

    # Value formula
    if cat=="annual":
        ref=f"B{ann_rows_inf[fname]}"
        style_formula(ws10.cell(r,3), f"={ref}", bg=cat_bg[cat])
    elif cat=="quarterly":
        parts=fname.rsplit("_Q",1); vn=parts[0]; qn=int(parts[1])
        addr=q_cells_inf.get((vn,qn),"")
        style_formula(ws10.cell(r,3), f"={addr}" if addr else '=""', bg=cat_bg[cat])
    elif cat=="interaction":
        parts=fname.split("_x_",1); v1p,v2p=parts[0],parts[1]
        addr=intr_cells_inf.get((v1p,v2p),"")
        style_formula(ws10.cell(r,3), f"={addr}" if addr else '=""', bg=cat_bg[cat])
    elif cat=="yield-lag":
        style_formula(ws10.cell(r,3), "=B39", bg=cat_bg[cat])

    style_label(ws10.cell(r,4), cat, bg=F_WHITE)
    ws10.row_dimensions[r].height=14

last_feat_r = 44+len(all_feats)-1

# ─── F. Prediction output ────────────────────────────────────────────────────
pr=last_feat_r+2
sec(ws10, pr, "SECTION F — Prediction Output  (paste Python model.predict() result here)", bg=F_PURPLE, cols="A:E")

out_rows=[
    ("log_yield_predicted","← PASTE Python model output here (model.predict(features)[0])",None,"Raw log-space output"),
    ("yield_kg_ha","EXP(log_yield_predicted) - 1  [inverse of LN(1+x)]",f"=IFERROR(EXP(B{pr+1})-1,\"\")","kg/ha"),
    ("total_production_kg","yield_kg_ha × luas_lahan_ha",f"=IFERROR(B{pr+2}*B40,\"\")","kg"),
    ("total_production_ton","total_production_kg ÷ 1000",f"=IFERROR(B{pr+3}/1000,\"\")","ton"),
]
for i,(lbl,hint,fval,unit) in enumerate(out_rows):
    r=pr+1+i
    style_label(ws10.cell(r,1),lbl,bold=True)
    if fval is None:
        c=ws10.cell(r,2); c.value="← PASTE HERE"
        c.font=Font(color="CC0000",name="Arial",size=9,bold=True)
        c.fill=fill("FADBD8"); c.border=thin()
        c.alignment=Alignment(horizontal="left",vertical="center")
    else:
        style_formula(ws10.cell(r,2),fval,bg=F_PFORM)
        ws10.cell(r,2).number_format="#,##0.00"
    style_label(ws10.cell(r,3),hint,bg=F_WHITE)
    style_label(ws10.cell(r,4),unit,bg=F_WHITE)
    ws10.row_dimensions[r].height=20

# Col widths for INFERENCE
set_w(ws10,1,45); set_w(ws10,2,24); set_w(ws10,3,45); set_w(ws10,4,16)

# ============================================================
# Save
# ============================================================
out=("./coffee_yield_pipeline.xlsx")
wb.save(out)
print(f"Saved → {out}")
print(f"Sheets: {[s.title for s in wb.worksheets]}")

Saved → ./coffee_yield_pipeline.xlsx
Sheets: ['LEGEND', 'RAW_CLIMATE', 'RAW_PRODUCTION', 'PRODUCTION_CALC', 'ANNUAL_CLIMATE', 'QUARTERLY_CLIMATE', 'CLIMATE_FEATURES', 'PREV_YIELD', 'TRAINING_TABLE', 'VALIDATION_TABLE', 'INFERENCE']


### For coffee_yield_pipeline.xlsx

In [ ]:
# AUTO FETCH
import joblib
import pandas as pd

# Configuration
EXCEL_FILE = '../../code/coffee_yield_pipeline.xlsx'
SHEET_NAME='INFERENCE'
MODEL_PATH = '../result/best_model.pkl'
FEATURES_PATH = '../result/features.csv'

# Feature names in EXACT order (D44:D94)
FEATURE_NAMES = [
    'rainfall_mm','temperature_celsius','relative_humidity_percent',
    'soil_moisture_percent','wind_speed_10m','dtr_celsius','vpd_kpa',
    'net_solar_rad_kwh_m2','rainfall_mm_Q1','temperature_celsius_Q1',
    'relative_humidity_percent_Q1','soil_moisture_percent_Q1',
    'wind_speed_10m_Q1','dtr_celsius_Q1','vpd_kpa_Q1',
    'net_solar_rad_kwh_m2_Q1','rainfall_mm_Q2','temperature_celsius_Q2',
    'relative_humidity_percent_Q2','soil_moisture_percent_Q2',
    'wind_speed_10m_Q2','dtr_celsius_Q2','vpd_kpa_Q2',
    'net_solar_rad_kwh_m2_Q2','rainfall_mm_Q3','temperature_celsius_Q3',
    'relative_humidity_percent_Q3','soil_moisture_percent_Q3',
    'wind_speed_10m_Q3','dtr_celsius_Q3','vpd_kpa_Q3',
    'net_solar_rad_kwh_m2_Q3','rainfall_mm_Q4','temperature_celsius_Q4',
    'relative_humidity_percent_Q4','soil_moisture_percent_Q4',
    'wind_speed_10m_Q4','dtr_celsius_Q4','vpd_kpa_Q4',
    'net_solar_rad_kwh_m2_Q4','temperature_celsius_x_vpd_kpa',
    'temperature_celsius_x_soil_moisture_percent',
    'temperature_celsius_x_dtr_celsius',
    'temperature_celsius_x_rainfall_mm',
    'vpd_kpa_x_soil_moisture_percent','vpd_kpa_x_dtr_celsius',
    'vpd_kpa_x_rainfall_mm','soil_moisture_percent_x_dtr_celsius',
    'soil_moisture_percent_x_rainfall_mm',
    'dtr_celsius_x_rainfall_mm','prev_yield_kg_ha'
]

def main():
    model = joblib.load(MODEL_PATH)
    feats = pd.read_csv(FEATURES_PATH)['feature'].tolist()
    
    try:
        df = pd.read_excel(EXCEL_FILE, sheet_name=SHEET_NAME, header=None)
    except ValueError as e:
        print(f"{e}")
        return
    
    # Extract values from column D (index 3), rows 44-94 (index 43-93)
    data = pd.DataFrame([df.iloc[43:94, 3].tolist()], columns=FEATURE_NAMES)
    
    # Check for missing/NaN values
    missing = data.columns[data.isna().any()].tolist()
    if missing:
        print(f"Warning!!! Fill the excel first!")
        print(f"{len(missing)} missing features: {missing}")
        return
    
    print(f"log_yield = {model.predict(data.reindex(columns=feats))[0]:.6f}")
    
if __name__ == "__main__":
    main()

log_yield = 6.583116
